# 0 — Подготовка датасета (единственный ноутбук подготовки)

Читает исходные данные **или** генерирует синтетику и собирает ЕДИНЫЙ артефакт `data/dataset`,
который читают все остальные ноутбуки. Других вариантов данных в `data/` нет (есть только сырьё
`data/sites/`).

* **real_objects** — читает сырые пиклы циклических опытов из `data/sites/` (подпапки
  «Потенциал разжижения» / «Штормовое разжижение» / «Сейсмо»), извлекает PPR(N), **девиатор q(N)**
  и **деформацию ε(N)**, сглаживает огибающую, парсит ведомость и обогащает физикой (CRR, Vs,
  PLAXIS). Вся логика — в `liquefaction_ai.data` (один вызов `build_real_objects_population`).
* **synthetic** — `generate_population` (генератор популяции).

Девиатор и деформация сохраняются в артефакт (`q_obs`/`eps_obs`), поэтому топология (ноутбук `4_2`)
работает на данных проекта **без повторного парсинга** сырых пиклов.

## Окружение

In [1]:
import os, sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai import generate_population, get_default_config, save_population_artifact, set_global_seed
from liquefaction_ai.data import (build_real_objects_population, find_cloud_root, DEFAULT_TEST_TYPES,
                                  load_population_artifact, prepare_benchmark_dataset)
from liquefaction_ai.viz import register_theme

register_theme()
SAVE_FIGS = True
config = get_default_config()
set_global_seed(config.seed)
if os.environ.get("LIQ_QUICK"):
    from dataclasses import replace
    config = replace(config, n_scenarios=1200, benchmark_subset=800)

DATASET_DIR = REPO_ROOT / "data" / "dataset"   # ЕДИНСТВЕННЫЙ артефакт данных проекта
SITES_DIR = REPO_ROOT / "data" / "sites"       # сырые пиклы циклических опытов (источник)
print("REPO_ROOT:", REPO_ROOT, "| seq_len:", config.seq_len)

REPO_ROOT: /Users/nikita/Desktop/projects/liquefaction-ai | seq_len: 72


## Шаг 1. Выбор источника

In [2]:
# >>> ЕДИНСТВЕННАЯ НАСТРОЙКА: что готовим <<<
SOURCE = os.environ.get("LIQ_DATASET", "real_objects")     # "synthetic" | "real_objects"
assert SOURCE in ("synthetic", "real_objects")
MAX_OBJECTS = int(os.environ.get("LIQ_REAL_MAXOBJ", "0")) or None   # лимит объектов/тип (0 = все)
print("Готовим датасет:", SOURCE, "→", DATASET_DIR)

Готовим датасет: real_objects → /Users/nikita/Desktop/projects/liquefaction-ai/data/dataset


## Шаг 2. Сборка единого артефакта `data/dataset`

In [3]:
# Сборка ЕДИНОГО артефакта data/dataset из исходных данных.
# Вся «грязная» логика (распаковка пиклов, извлечение PPR/девиатора/деформации, сглаживание
# огибающей, парсинг ведомости, подгонка CRR по ИГЭ, физическое обогащение) — в пакете
# liquefaction_ai.data (raw_loader / real_adapter / soil_profile / grainsize / ppr_envelope).
if SOURCE == "synthetic":
    population = generate_population(config)
    print("Синтетическая популяция сгенерирована:", len(population["meta"]), "образцов")
else:
    raw_root = find_cloud_root([os.environ.get("LIQ_REAL_ROOT", ""), SITES_DIR,
                                REPO_ROOT.parent / "Облако разжижения"])
    if raw_root is None:
        raise FileNotFoundError(
            f"Сырые пиклы не найдены. Положите опыты в {SITES_DIR} "
            f"(подпапки {DEFAULT_TEST_TYPES}) или задайте LIQ_REAL_ROOT.")
    print("Читаю сырые опыты из:", raw_root)
    population = build_real_objects_population([(raw_root, DEFAULT_TEST_TYPES)], config,
                                              max_objects=MAX_OBJECTS)
    m = population["meta"]
    print("Собрано:", len(m), "образцов |", m["object"].nunique(), "объектов |",
          "разжижение:", round(float(population["liq_label"].mean()), 3),
          "| есть q/ε:", "q_obs" in population and "eps_obs" in population)

from dataclasses import replace
config = replace(config, dataset_source=SOURCE)   # артефакт помечается реальным источником, а не дефолтным 'synthetic'
save_population_artifact(DATASET_DIR, population, config)
print("Сохранено в", DATASET_DIR, "— все ноутбуки (1_*, 2_*, 3_*, 4_*) читают этот каталог.")

Читаю сырые опыты из: /Users/nikita/Desktop/projects/liquefaction-ai/data/sites
Собрано: 996 образцов | 20 объектов | разжижение: 0.466 | есть q/ε: True
Сохранено в /Users/nikita/Desktop/projects/liquefaction-ai/data/dataset — все ноутбуки (1_*, 2_*, 3_*, 4_*) читают этот каталог.


## Шаг 3. Проверка

In [4]:
# Проверка готового артефакта и benchmark-сплита (site-held-out по объекту).
import torch
population, loaded_config = load_population_artifact(DATASET_DIR)
benchmark = prepare_benchmark_dataset(population, loaded_config, torch.device("cpu"))
for split_name in ["train", "val", "test"]:
    s = benchmark[split_name]
    print(f"{split_name:6s}: static={tuple(s['static'].shape)}, seq={tuple(s['seq_in'].shape)}, "
          f"liq_rate={float(s['label'].float().mean()):.3f}")
meta = population["meta"]
display(pd.DataFrame({
    "Показатель": ["источник", "образцов", "объектов", "доля разжижения", "средний N_liq",
                   "длина seq", "статич. признаков", "девиатор q сохранён", "деформация ε сохранена",
                   "grouped split (site-held-out)"],
    "Значение": [SOURCE, len(meta), int(meta["object"].nunique()) if "object" in meta else "—",
                 round(float(meta["liq_label"].mean()), 4), round(float(meta["N_liq_true"].mean()), 1),
                 population["r_obs"].shape[1], population["static_features"].shape[1],
                 "q_obs" in population, "eps_obs" in population,
                 bool(getattr(loaded_config, "group_split_by_object", False))],
}))

train : static=(675, 34), seq=(675, 72, 5), liq_rate=0.449
val   : static=(140, 34), seq=(140, 72, 5), liq_rate=0.521
test  : static=(181, 34), seq=(181, 72, 5), liq_rate=0.464


,Показатель,Значение
0,источник,real_objects
1,образцов,996
2,объектов,20
3,доля разжижения,0.4659
4,средний N_liq,1607.8
5,длина seq,72
6,статич. признаков,34
7,девиатор q сохранён,True
8,деформация ε сохранена,True
9,grouped split (site-held-out),True
